In [0]:
from pyspark.sql.functions import *

changed_regions_df = (spark.read.format('csv')
                        .option('header', 'true')
						.schema('id INT, name STRING, date STRING')
                        .load('/Volumes/nyctaxi/landing/regions')
			).withColumn('load_timestamp', current_timestamp())
changed_regions_df.display()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit, coalesce
from pyspark.sql.types import TimestampType

# Load target Delta table
dt = DeltaTable.forName(spark, "NYCTAXI.SILVER.REGIONS_SCD_2")

# Current active rows (effective_end_date IS NULL)
current_active = dt.toDF().filter(col("effective_end_date").isNull())

# Rename source columns to avoid ambiguity
src = (
    changed_regions_df
    .withColumnRenamed("name", "src_name")
    .withColumnRenamed("date", "src_date")
    .withColumnRenamed("load_timestamp", "src_load_timestamp")  # avoid clash
)

tgt = current_active.withColumnRenamed("name", "tgt_name")

# Detect changes: new IDs or attribute differences
changes_only = (
    src.join(tgt, on="id", how="left")
       .filter(
           (col("tgt_name").isNull()) |
           (coalesce(col("src_name"), lit('')) != coalesce(col("tgt_name"), lit('')))
       )
       .dropDuplicates(["id"])
)

# Step 1: Expire old versions
dt.alias("t").merge(
    source=changes_only.alias("s"),
    condition="t.id = s.id AND t.effective_end_date IS NULL"
).whenMatchedUpdate(
    set={"t.effective_end_date": current_timestamp()}
).execute()

# Step 2: Insert new versions
new_versions = (
    changes_only
    .withColumn("effective_start_date", col("src_date").cast(TimestampType()))
    .withColumn("effective_end_date", lit(None).cast(TimestampType()))
    .withColumn("load_timestamp", current_timestamp())  # fresh ingestion time
    .select(
        "id",
        col("src_name").alias("name"),
        "effective_start_date",
        "effective_end_date",
        "load_timestamp"
    )
)

new_versions.write.mode("append").saveAsTable("NYCTAXI.SILVER.REGIONS_SCD_2")
